# Context-management cost analysis -- sprint 4 (bd polyglot-pdf-6x1y)

Interactive analysis of the bd-`wyhh` three-arm matrix: **2 models x {unbounded,
section-reset, sliding-window} x {happy, unhappy} x 20 questions** against live
Gemini, driven through the real production stack (bd `4loz`).

**Trust gate:** every computation comes from `cost_matrix_lib.py`, reviewed
adversarially (the library over 3 rounds; this notebook's render code + the
section-4 drop classifier in their own passes). The notebook only loads,
models, and renders.

**Schema scope (read this before the charts):** the measurement harness fills
**20 questions spanning the first 3 sections** of the demo schema
(client-info 12, reasons 5, living-situation 3) -- NOT the full 15-section,
174-field production form. So the section-reset arm shows **2 real restarts**
(at ~q12 and ~q17), not the ~14 a whole form would have. Whole-form numbers in
sections 7-9 are MODELED extrapolations from this measured base, clearly
labeled as such.

**Data caveats baked into the loaders** (encoded as `valid`/`degraded` flags,
not footnotes): the two 3.1 single-session unhappy cells were terminated by the
session-duration GoAway at ~q12-13; one 2.5 section-reset cell lost q16-17 to a
`1006` transient (recovered q18-20); 2.5 cells carry occasional silent
`update_form_fields` TypeErrors (bd `d24n`).

In [1]:
import sys, json
from pathlib import Path
for cand in (Path.cwd(), Path.cwd() / 'scripts' / 'spikes'):
    if (cand / 'cost_matrix_lib.py').exists():
        sys.path.insert(0, str(cand)); break
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import cost_matrix_lib as L

questions, turns, curves, runs = L.load_matrix()     # validates + calibrates
print(f"loaded {len(questions)} question rows / {len(turns)} turns / 12 runs")
print(f"calibration: {L.CHAR_RATE:.4f} tok/char + {L.FIXED_OVERHEAD:.0f} tok overhead")
live = L.recompute_si_scenarios()
drift = {k: live[s] - v for (k, v), s in zip(L.SI_SCENARIOS_CHARS.items(), ('full','jit','drip'))}
print('SI literal drift vs live recompute (chars):', drift, '+ sections:',
      [a-b for a, b in zip(live['sections'], L.SECTION_SI_CHARS)])

loaded 240 question rows / 720 turns / 12 runs
calibration: 0.2798 tok/char + 992 tok overhead


2026-06-12 19:34:03

 [

info     

] 

...

0

SI literal drift vs live recompute (chars):

{"full-dump (today's no-sections port)": 0, 'jit-manifest (built, unmeasured live)': 0, 'drip-in (fields=[]; schema via tools)': 0}

+ sections:

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

## 1. Data integrity panel

Run-level caveats first -- every later chart excludes invalid (dead-session)
questions unless explicitly labeled otherwise.

In [2]:
summary = L.arm_summary(questions, runs)
display(summary[['model','arm','path','questions_counted','total_tokens','total_cost',
                 'text_in_share','confirmed_fields','n_stalled_turns','truncated']]
        .style.format({'total_cost': '${:.3f}', 'text_in_share': '{:.1%}'}))
print('truncated = session killed mid-run (GoAway / 1006); questions_counted is the valid basis')

,model,arm,path,questions_counted,total_tokens,total_cost,text_in_share,confirmed_fields,n_stalled_turns,truncated
0,2.5,section-reset,happy,20,213867,$0.188,95.0%,16,0,False
1,2.5,sliding-window,happy,20,534428,$0.330,98.0%,15,0,False
2,2.5,unbounded,happy,20,535613,$0.338,97.9%,13,0,False
3,2.5,section-reset,unhappy,18,444425,$0.323,96.4%,15,11,True
4,2.5,sliding-window,unhappy,20,1342518,$0.797,97.9%,19,0,False
5,2.5,unbounded,unhappy,20,1317634,$0.781,98.0%,18,0,False
6,3.1,section-reset,happy,20,341266,$0.627,58.1%,19,0,False
7,3.1,sliding-window,happy,20,828400,$1.023,81.2%,20,0,False
8,3.1,unbounded,happy,20,1047190,$1.624,67.0%,18,0,False
9,3.1,section-reset,unhappy,20,840219,$1.729,46.9%,20,0,False


truncated = session killed mid-run (GoAway / 1006); questions_counted is the valid basis

## 2. Where the money goes -- cost sources

The four billable streams per arm (valid questions only).

- **2.5 is one-note: text input is 95-98% of every arm's cost** -- the
  re-billed system instruction + accumulated conversation. Audio is negligible.
- **3.1 is a different animal: AUDIO INPUT dominates.** 3.1 retains and
  re-bills the conversation AUDIO in its context (text-out is literally 0 --
  it is an audio-native path), so its audio-input tokens compound to ~50x 2.5's
  (verified next cell). This -- not thinking tokens -- is why 3.1 costs 3-5x
  2.5. The chart is correctly labeled; the audio-in bars for 3.1 are real.

In [3]:
src = summary.melt(
    id_vars=['model','arm','path'],
    value_vars=['text_in_tokens','audio_in_tokens','text_out_tokens','audio_out_tokens'],
    var_name='stream', value_name='tokens')
_RATE_KEY = {'text_in_tokens':'text_input','audio_in_tokens':'audio_input',
             'text_out_tokens':'text_output','audio_out_tokens':'audio_output'}
src['cost'] = [t * L.RATES[m][_RATE_KEY[s]] / 1e6
               for m, s, t in zip(src['model'], src['stream'], src['tokens'])]
fig = px.bar(src, x='arm', y='cost', color='stream', facet_col='path', facet_row='model',
             title='Cost by billable stream (valid questions, 20-question runs)',
             labels={'cost': 'USD'}, height=560)
fig.show()

# Make the 3.1 audio-context finding explicit: per-question audio-in growth.
ai = (questions[questions['valid']]
      .pivot_table(index='q', columns='model', values='audio in tokens', aggfunc='mean'))
print('Per-question AUDIO-IN tokens (mean over arms/paths), q1 vs q20:')
print(f"  2.5: {ai['2.5'].iloc[0]:.0f} -> {ai['2.5'].iloc[-1]:.0f}")
print(f"  3.1: {ai['3.1'].iloc[0]:.0f} -> {ai['3.1'].iloc[-1]:.0f}  "
      f"(~{ai['3.1'].iloc[-1]/max(ai['2.5'].iloc[-1],1):.0f}x 2.5 -- audio context re-billing)")

Per-question AUDIO-IN tokens (mean over arms/paths), q1 vs q20:

  2.5: 64 -> 1000

  3.1: 2579 -> 15308  (~15x 2.5 -- audio context re-billing)

## 3. Cumulative cost race, question by question

The 20-question measured race per arm. Slopes: unbounded steepens (context
compounds), section-reset saw-tooths shallow (resets at the 2 boundaries), the
sliding-window arm tracks unbounded almost exactly -- the 12k trigger over an
~8.2k unevictable instruction floor leaves little to evict, and **the window
never actually fired in this matrix** (section 6).

In [4]:
v = questions[questions['valid']].sort_values('q').copy()
v['cum_cost'] = v.groupby(['model','arm','path'])['total cost'].cumsum()
fig = px.line(v, x='q', y='cum_cost', color='arm', facet_col='path', facet_row='model',
              markers=True, height=560, labels={'cum_cost': 'cumulative USD', 'q': 'question'},
              title='Measured cumulative cost (invalid/dead questions excluded)')
fig.show()

## 4. Context length per billed event -- with the drops correctly classified

This is the raw "how much memory is the model carrying" trace, and the drops
in it come from THREE distinct mechanisms that an earlier version of this chart
conflated. The library now classifies each drop by **where it lands** relative
to the run's own SI floor (`cost_matrix_lib._classify`, reviewed):

- **session_restart** (orange) -- the curve falls back near a *section* SI base
  (~1,840-2,700). Only the section-reset arm. On 2.5 these are the 2 real
  section boundaries at ~event 26 and ~37.
- **audio_context_dip** (purple) -- **3.1 only**, in *every* arm: 3.1 re-bills
  conversation audio, so its prompt sawtooths ~every 4 events even with NO
  compression and NO restart. These are NOT evictions and NOT restarts; they
  are intrinsic 3.1 behavior, and mislabeling them as evictions was the bug in
  the prior chart.
- **compression_eviction** (red) -- a real sliding-window eviction. **There are
  ZERO in this matrix:** at trigger=12k/window=9k the window never fired (it
  sits just above the ~8.2k SI floor with little headroom). That absence is
  itself the headline -- the window did nothing.
- **billing_noise** (gray, hidden by default) -- sub-400-token metering jitter,
  not a real context change.

The lines per arm are the turn-memory-vs-context-length distribution; the
markers say which mechanism (if any) moved the context.

In [5]:
order = [('3.1','happy'),('3.1','unhappy'),('2.5','happy'),('2.5','unhappy')]
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{m} / {p}' for m, p in order],
                    shared_xaxes=False)
arm_color = {'unbounded':'#1f77b4','section-reset':'#2ca02c','sliding-window':'#ff7f0e'}
mark_color = {'session_restart':'orange','audio_context_dip':'purple',
              'compression_eviction':'red','billing_noise':'lightgray'}
shown_arm, shown_mark = set(), set()
for i, (m, p) in enumerate(order):
    r, c = i // 2 + 1, i % 2 + 1
    for arm in ['unbounded','section-reset','sliding-window']:
        d = curves[(curves['model']==m)&(curves['path']==p)&(curves['arm']==arm)].sort_values('event_index')
        if not len(d): continue
        fig.add_trace(go.Scatter(x=d['event_index'], y=d['prompt_tokens'], mode='lines',
                                 line=dict(color=arm_color[arm]), name=arm,
                                 legendgroup=arm, showlegend=arm not in shown_arm), row=r, col=c)
        shown_arm.add(arm)
    dd = curves[(curves['model']==m)&(curves['path']==p)&(curves['drop'])
                &(curves['drop_kind']!='billing_noise')]
    for kind, grp in dd.groupby('drop_kind'):
        fig.add_trace(go.Scatter(x=grp['event_index'], y=grp['prompt_tokens'], mode='markers',
                                 marker=dict(color=mark_color[kind], size=9, symbol='x'),
                                 name=kind, legendgroup=kind, showlegend=kind not in shown_mark),
                      row=r, col=c)
        shown_mark.add(kind)
fig.update_layout(height=640, title='Prompt (context) tokens per billed event, drops classified by mechanism')
fig.update_xaxes(title_text='billed event #'); fig.update_yaxes(title_text='prompt tokens')
fig.show()
print('drop mechanism counts across the matrix:')
print(curves[curves['drop']].groupby(['model','drop_kind']).size().unstack(fill_value=0))

drop mechanism counts across the matrix:

drop_kind  audio_context_dip  billing_noise  session_restart
model                                                       
2.5                        0              4                4
3.1                       56              0                7

## 5. Latency -- with a load-bearing caveat

**This latency result is NOT production-representative and must not be read as
one.** It measures `activity_end -> first audio byte` for *injected synthetic
audio* over a *local* backend -- it strips out the real audio round-trip and
network path where 3.1's slowness actually shows. On this narrow harness 2.5
and 3.1 look near-parity; the earlier LIVE-DEMO finding that **3.1 is much
slower end-to-end stands** and is the production-relevant one. What this chart
does support: **eviction events add no latency** (and there were no real
evictions anyway), and section restarts add their own boundary pause (the
restart greeting wait, not shown here).

In [6]:
lt = turns.dropna(subset=['latency_s'])
fig = px.box(lt, x='arm', y='latency_s', color='model', facet_col='path', height=480,
             title='Turn latency: activity-end to first audio byte (s) -- LOCAL harness, not production')
fig.show()
display(L.latency_summary(turns).style.format({'median': '{:.2f}s', 'p90': '{:.2f}s'}))

,model,arm,path,n,median,p90
0,2.5,section-reset,happy,39,2.17s,3.99s
1,2.5,section-reset,unhappy,68,2.56s,3.51s
2,2.5,sliding-window,happy,39,1.75s,2.64s
3,2.5,sliding-window,unhappy,79,1.73s,2.78s
4,2.5,unbounded,happy,39,1.81s,2.90s
5,2.5,unbounded,unhappy,79,1.61s,2.62s
6,3.1,section-reset,happy,40,1.84s,2.19s
7,3.1,section-reset,unhappy,80,1.75s,2.05s
8,3.1,sliding-window,happy,40,1.88s,2.08s
9,3.1,sliding-window,unhappy,52,1.80s,1.95s


## 6. Honesty check: the window model vs what actually happened

Before any what-if chart, the model faces the measured cells. The model is
**best-case** (server evicts exactly as configured). Measured reality at
trigger=12k/window=9k: **the window never fired a real eviction** (section 4:
zero `compression_eviction` events) -- so measured sliding-window cost tracks
unbounded, and for 2.5-unhappy even edged slightly above it (pure no-op +
metering noise). At SMALL triggers over a minimal SI (1.5k/800), the h04c
matrix measured **reliable** firing (3/3, both models, 20-turn runs) -- so the
lean-SI what-ifs below assume the reliable regime, a STATED assumption, not a
measured fact at these configs.

In [7]:
growth = L.conversation_growth_per_turn(curves, runs)
display(growth.style.format({'growth_per_question': '{:.0f}'}))
check = L.window_model_check(questions, runs, growth)
display(check.style.format({c: '${:.3f}' for c in ('modeled_text_in','measured_text_in','unbounded_text_in')}))
print('Read: modeled = best-case eviction. measured ~ unbounded here because the')
print('12k/9k window never fired (section 4); the 2.5-unhappy row even sits slightly')
print('ABOVE unbounded -- compression at this config over the full-dump floor is a no-op.')

,model,path,growth_per_event,events_per_question,growth_per_question
0,2.5,happy,233.000000,2.000000,466
1,2.5,unhappy,218.000000,4.000000,872
2,3.1,happy,736.000000,2.000000,1472
3,3.1,unhappy,766.000000,3.833333,2936


,model,path,n_questions,modeled_text_in,measured_text_in,unbounded_text_in,measured_evictions
0,2.5,happy,20,$0.210,$0.262,$0.262,0
1,2.5,unhappy,20,$0.423,$0.657,$0.646,2
2,3.1,happy,20,$0.323,$0.504,$0.526,4
3,3.1,unhappy,12,$0.374,$0.520,$0.538,16


Read: modeled = best-case eviction. measured ~ unbounded here because the

12k/9k window never fired (section 4); the 2.5-unhappy row even sits slightly

ABOVE unbounded -- compression at this config over the full-dump floor is a no-op.

## 7. The lever: what the system instruction floor does to a sliding window

Three schema-feeding scenarios, all built from the real demo schema + real
backend templates (sizes recomputed live in cell 1):

| scenario | SI chars | unevictable floor (tokens) |
|---|---|---|
| full dump (today's no-sections port) | 25,632 | ~8,160 |
| JIT manifest (already built; never measured live) | 10,204 | ~3,850 |
| drip-in (`fields=[]`; schema arrives via tool responses, EVICTABLE) | 3,069 | ~1,850 |

The design point (maintainer, 2026-06-12): the giant dump is NOT an inherent
cost of windowing -- the exploration tools (`get_field_value`,
`get_next_field_schema_for_jit`) let the model *discover* field parameters and
dripped-in schema ages out of the window naturally. **The whole-form curves
below are MODELED** (174 questions, 15 sections) from the measured 20-question
base -- extrapolations, not measurements.

In [8]:
N_FULL_FORM = 174
schema = json.loads(L.SCHEMA_PATH.read_text(encoding='utf-8'))
sec_counts = {}
for f in schema['fields']:
    sec_counts[f.get('section_id') or 's1'] = sec_counts.get(f.get('section_id') or 's1', 0) + 1
QPS = list(sec_counts.values())
G = {(r_['model'], r_['path']): (r_['growth_per_question'], r_['events_per_question'])
     for _, r_ in growth.iterrows()}

def whole_form_curves(model, path, retained_grid=range(250, 6001, 250)):
    g, epq = G[(model, path)]
    rows = []
    for label, chars in L.SI_SCENARIOS_CHARS.items():
        for retained in retained_grid:
            base = L.si_tokens_from_chars(chars)
            window = base + retained
            trigger = int(window * 1.3)
            df = L.window_cost_curve(chars, g, N_FULL_FORM, model, [(trigger, int(window))],
                                     events_per_q=epq)
            rows.append({'scenario': label, 'retained_memory_tokens': retained,
                         'text_in_cost_174q': df['cum_text_in_cost'].iloc[-1]})
    out = pd.DataFrame(rows); out['model'], out['path'] = model, path
    return out

wf = pd.concat([whole_form_curves('2.5', 'happy'), whole_form_curves('2.5', 'unhappy')])
g_h, epq_h = G[('2.5', 'happy')]; g_u, epq_u = G[('2.5', 'unhappy')]
sr_h = L.section_reset_model_cost(g_h, QPS, '2.5', events_per_q=epq_h)
sr_u = L.section_reset_model_cost(g_u, QPS, '2.5', events_per_q=epq_u)
fig = px.line(wf, x='retained_memory_tokens', y='text_in_cost_174q', color='scenario',
              facet_col='path', height=480, log_y=True,
              title='MODELED whole form (174 q, 2.5): text-in cost vs retained conversation memory',
              labels={'retained_memory_tokens': 'retained conversation (tokens kept after eviction)',
                      'text_in_cost_174q': 'text-in USD (174 q, log scale)'})
for i, sr in enumerate((sr_h, sr_u), start=1):
    fig.add_hline(y=sr, line_dash='dash', line_color='black', col=i,
                  annotation_text=f'section-reset flow: ${sr:.2f}', annotation_position='top left')
fig.show()
print(f'section-reset modeled whole-form text-in: happy ${sr_h:.2f} / unhappy ${sr_u:.2f}')
print('NOTE: best-case model; +audio/output adds a roughly arm-independent constant.')

section-reset modeled whole-form text-in: happy $1.61 / unhappy $5.18

NOTE: best-case model; +audio/output adds a roughly arm-independent constant.

### Reading the elbow

Cost is nearly **flat in retained memory** and **steps hugely with the SI
scenario** -- the floor, not the window, is the lever. No sharp elbow to hunt:
pick retained memory for *behavioral* reasons (~1,500-2,000 tokens is roughly
the last 3-6 exchanges) and spend the engineering effort on the SI diet. With
an 80/20 happy/unhappy mix at 174 questions, retained 1,750 tokens (2.5,
text-in, best-case -- the table below is the source of record): section resets
~$2.32; full-dump window ~$2.38 (NO better than sections); JIT window ~$1.34;
drip window ~$0.86.

In [9]:
MIX = 0.8
mix_rows = []
for label, chars in L.SI_SCENARIOS_CHARS.items():
    base = L.si_tokens_from_chars(chars)
    window = int(base + 1750); trigger = int(window * 1.3)
    ch = L.window_cost_curve(chars, g_h, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_h)['cum_text_in_cost'].iloc[-1]
    cu = L.window_cost_curve(chars, g_u, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_u)['cum_text_in_cost'].iloc[-1]
    mix_rows.append({'flow': f'sliding window / {label}', 'config': f'trigger {trigger:,} / window {window:,}',
                     'happy_174q': ch, 'unhappy_174q': cu, 'mix_80_20': MIX*ch + (1-MIX)*cu})
mix_rows.append({'flow': 'section-reset (current production)', 'config': '15 restarts',
                 'happy_174q': sr_h, 'unhappy_174q': sr_u, 'mix_80_20': MIX*sr_h + (1-MIX)*sr_u})
display(pd.DataFrame(mix_rows).sort_values('mix_80_20')
        .style.format({c: '${:.2f}' for c in ('happy_174q','unhappy_174q','mix_80_20')}))

,flow,config,happy_174q,unhappy_174q,mix_80_20
2,sliding window / drip-in (fields=[]; schema via tools),"trigger 4,680 / window 3,600",$0.72,$1.44,$0.86
1,"sliding window / jit-manifest (built, unmeasured live)","trigger 7,274 / window 5,596",$1.12,$2.24,$1.34
3,section-reset (current production),15 restarts,$1.61,$5.18,$2.32
0,sliding window / full-dump (today's no-sections port),"trigger 12,886 / window 9,913",$1.98,$3.96,$2.38


## 8. Interactive explorer (live kernel only)

Sliders: happy/unhappy mix and retained memory; recomputes the comparison
table. (Static defaults above are the committed record; needs a running
kernel.)

In [10]:
try:
    import ipywidgets as W
    @W.interact(mix=W.FloatSlider(value=0.8, min=0.0, max=1.0, step=0.05, description='happy share'),
                retained=W.IntSlider(value=1750, min=250, max=6000, step=250, description='retained tok'))
    def explore(mix=0.8, retained=1750):
        rows = []
        for label, chars in L.SI_SCENARIOS_CHARS.items():
            base = L.si_tokens_from_chars(chars)
            window = int(base + retained); trigger = int(window * 1.3)
            ch = L.window_cost_curve(chars, g_h, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_h)['cum_text_in_cost'].iloc[-1]
            cu = L.window_cost_curve(chars, g_u, N_FULL_FORM, '2.5', [(trigger, window)], events_per_q=epq_u)['cum_text_in_cost'].iloc[-1]
            rows.append({'flow': label, 'mix': mix*ch + (1-mix)*cu})
        rows.append({'flow': 'section-reset', 'mix': mix*sr_h + (1-mix)*sr_u})
        display(pd.DataFrame(rows).sort_values('mix').style.format({'mix': '${:.2f}'}))
except Exception as e:
    print('ipywidgets unavailable in this environment:', e)

interactive(children=(FloatSlider(value=0.8, description='happy share', max=1.0, step=0.05), IntSlider(value=1…

## 9. Cost/benefit and recommendations

### What the measured data settles
1. **Text-in is the bill on 2.5** (95-98%); **audio-in is the bill on 3.1**
   (re-billed conversation audio, ~50x 2.5, the reason 3.1 costs 3-5x). Both
   point the same way: the model carrying context is the cost.
2. **Section resets beat the naive single-session port at any scale** --
   measured at 20 q ($0.19 vs $0.33-0.34 happy), modeled vs unbounded's
   $8.5-29 at 174 q.
3. **The 12k/9k sliding window did NOTHING** -- zero real evictions fired
   (section 4); it cannot evict the SI, which is the actual cost.
4. **No latency cost at evictions; the duration GoAway is a separate ~13-15
   min wall** any single-session flow must handle (`c5t` shipped, `l3a.2`
   policy open, `wnt` decided) regardless of cost.

### What the modeling (trusted code, best-case) indicates
5. **The schema-feeding strategy is worth 2-5x; window size is worth almost
   nothing past ~1.5-2k retained.** At 1,750 retained: full-dump ~$2.38 (no
   better than sections' ~$2.32 -- the naive port loses), JIT ~$1.34 (-42%),
   drip ~$0.86 (-63%) per 174-q form (80/20, text-in, 2.5).
6. **The hybrid is viable:** fewer/larger sections WITH a window inside each
   (the 54-field medical section is the section-reset flow's own pain point).

### DECISION (maintainer, 2026-06-12): shelve the sliding window; pivot to SI
reduction via task delegation. The window adds ~0-2% at every measured/modeled
size; the SI is the lever and compression cannot touch it. The next epic is
consolidated in the P0 planning bead **polyglot-pdf-pivot1** (live model
focused on the current field + a cheap flash-lite-class model validating
completed sections/forms, which lets the SI shrink toward the drip-in floor).

### Known limits
Best-case eviction model (firing reliability at lean-SI configs assumed from
small-trigger h04c data, not measured); 2.5-only calibration (3.1 what-ifs ~3%
optimistic); text-in only in the what-ifs; single run per cell (no variance
bars); latency is a local harness (NOT production); the GoAway-truncated cells
contribute only their live spans; the harness fills 3 of 15 sections (whole-
form figures are modeled extrapolations).